Una volta installati i packages necessari, è richiesto di fare un riavvio della sessione (no eliminazione dei dati). Questo è dovuto a diversi conflitti di dipendenze, riavviando la sessione si risolvono queste problematiche.

In [15]:
!pip install -q \
  "numpy<2" \
  llama-index \
  llama-index-llms-gemini \
  llama-index-llms-huggingface \
  llama-index-embeddings-huggingface \
  transformers \
  accelerate \
  bitsandbytes \
  rich \
  pandas

In [6]:
from google.colab import userdata
import os

GEMINI_API_KEY = userdata.get('GEMINI_KEY')


if GEMINI_API_KEY:
    print("Chiave Gemini trovata")
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
else:
    print("Chiave Gemini non trovata")

Chiave Gemini trovata


In [7]:
import os
import json
import torch
import pandas as pd
import re

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markdown import Markdown
from google.colab import userdata

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

console = Console()

# ===============================
# LLM SETUP
# ===============================


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    try:
        import google.generativeai as genai
        from llama_index.llms.gemini import Gemini

        genai.configure(api_key=GOOGLE_API_KEY)
        llm = Gemini(model="models/gemini-2.5-flash", temperature=0.1)
        console.print(Panel("✅ Gemini LLM attivo", style="green"))
    except Exception as e:
        console.print(f"⚠️ Gemini non disponibile: {e}", style="yellow")



# ===============================
# GENERAZIONE DOCUMENTI FINANZIARI (FITTIZI)
# ===============================
financial_documents = [
    {
        "title": "Q2 2024 - Report Finanziario Trimestrale",
        "content": """
        REPORT FINANZIARIO Q2 2024 - FinSecure Analytics S.p.A.

        SINTESI ESECUTIVA:
        - Ricavi totali: €120.5M (+8% YoY)
        - Costi operativi: €95.2M (+12% YoY)
        - EBITDA: €25.3M (margine 21%)
        - Utile netto: €12.8M

        ANALISI DEL RISCHIO:
        - Esposizione debitoria aumentata del 18% rispetto al Q1
        - Debito totale: €87.5M
        - Ratio debito/equity: 2.1 (soglia critica: 2.0)
        - Indice di liquidità corrente: 1.15 (sotto la soglia regolamentare di 1.25)
        - Concentrazione clienti: Top 3 clienti rappresentano il 67% dei ricavi

        FATTORI DI RISCHIO IDENTIFICATI:
        - Aumento dei tassi di interesse BCE (+0.5%)
        - Volatilità valutaria EUR/USD elevata
        - Ritardi nei pagamenti da parte di 2 clienti principali (€8.2M)
        - Mancata copertura assicurativa su crediti sopra €5M
        """
    },
    {
        "title": "Risk Assessment - Analisi Credito Q2 2024",
        "content": """
        VALUTAZIONE DEL RISCHIO CREDITO - Q2 2024

        PORTFOLIO CREDITI:
        - Crediti commerciali totali: €45.3M
        - Crediti scaduti >90 giorni: €8.7M (19.2%)
        - Accantonamenti per svalutazione crediti: €2.1M (insufficienti secondo IFRS 9)

        ESPOSIZIONI CRITICHE:
        - Cliente A (settore automotive): €15.2M - rating deteriorato da BBB a BB-
        - Cliente B (retail): €12.8M - procedura di ristrutturazione del debito in corso
        - Cliente C (costruzioni): €9.5M - nessun pagamento negli ultimi 75 giorni

        RACCOMANDAZIONI:
        - Incrementare accantonamenti a €4.5M (tasso di copertura 10%)
        - Attivare procedure di recupero crediti per Cliente C
        - Richiedere garanzie bancarie per nuovi ordini Cliente A e B
        - Diversificare il portafoglio clienti (concentrazione eccessiva)

        MANCANZE RILEVATE:
        - Assenza di sistema automatico di credit scoring
        - Monitoraggio rating clienti non aggiornato da 6 mesi
        - Policy di gestione crediti non allineata alle best practice IFRS 9
        """
    },
    {
        "title": "Market Risk Report - Analisi Volatilità Mercati",
        "content": """
        REPORT RISCHIO DI MERCATO - Giugno 2024

        ESPOSIZIONE VALUTARIA:
        - Fatturato in USD: 35% (€42M)
        - Costi in EUR: 95%
        - Hedging valutario: solo 40% dell'esposizione coperta
        - Perdita potenziale su esposizione non coperta: €3.2M (scenario stress +10% USD/EUR)

        ANALISI TASSI DI INTERESSE:
        - Debito a tasso variabile: €65M (74% del totale)
        - Debito a tasso fisso: €22.5M (26%)
        - Sensitivity: +1% tasso = -€650K utile annuo
        - Nessun interest rate swap attivo (raccomandato)

        VOLATILITÀ MATERIE PRIME:
        - Esposizione a semiconduttori: €18M annui
        - Incremento prezzi: +22% negli ultimi 6 mesi
        - Nessun contratto di fornitura a lungo termine
        - Impatto stimato marginalità: -3.5%

        STRESS TEST RISULTATI:
        - Scenario "Recessione Moderata": -€8.5M utile
        - Scenario "Crisi Liquidità": rischio insolvenza entro 9 mesi
        - Scenario "Shock Tassi +2%": EBITDA margin da 21% a 15%
        """
    },
    {
        "title": "Compliance Report - Audit Normativo Q2 2024",
        "content": """
        REPORT CONFORMITÀ NORMATIVA - Q2 2024

        STATO COMPLIANCE:
        - Regolamento GDPR: Conforme
        - Direttiva MiFID II: Conforme con riserva (1 non-conformità minore)
        - Basilea III (applicabile): NON CONFORME su ratio liquidità
        - IFRS 9 (impairment crediti): Parzialmente conforme

        NON-CONFORMITÀ CRITICHE:
        1. Liquidity Coverage Ratio (LCR): 115% (minimo 125%)
        2. Accantonamenti crediti: metodologia ECL non completamente implementata
        3. Stress testing: frequenza trimestrale non rispettata (ultimo test: 8 mesi fa)
        4. Reporting regolamentare: ritardo di 15 giorni su FINREP

        POLICY MANCANTI O INCOMPLETE:
        - Manca Policy formale di gestione rischio liquidità
        - Business Continuity Plan non aggiornato dal 2022
        - Procedura di escalation rischi operativi non documentata
        - Manca sistema di Three Lines of Defense formalmente implementato

        AZIONI CORRETTIVE RICHIESTE:
        - Implementare buffer di liquidità addizionale: €15M entro 60 giorni
        - Completare upgrade sistema impairment crediti entro Q3 2024
        - Schedulare stress test mensili dal Q3 2024
        - Nominare Compliance Officer dedicato (attualmente assente)
        """
    },
    {
        "title": "Operational Risk Assessment - Q2 2024",
        "content": """
        VALUTAZIONE RISCHI OPERATIVI - Q2 2024

        INCIDENTI REGISTRATI:
        - Totale incidenti: 12 (vs 7 in Q1)
        - Perdita operativa totale: €385K
        - Incidente più grave: errore di trading (€180K)

        RISK INDICATORS:
        - Turnover personale chiave: 18% (soglia critica: 12%)
        - Downtime sistemi IT: 14 ore (SLA: max 8 ore/trimestre)
        - Failed transactions: 0.8% (target: <0.5%)
        - Reclami clienti: +45% rispetto a Q1

        VULNERABILITÀ IDENTIFICATE:
        - Single point of failure su sistema core banking
        - Assenza backup geografico dei dati (solo locale)
        - Manca procedura formale di change management IT
        - Formazione staff su cyber security: solo 30% completata
        - Nessun disaster recovery test negli ultimi 18 mesi

        RISCHI EMERGENTI:
        - Dipendenza da fornitore unico per servizi cloud (AWS)
        - Sistemi legacy non aggiornati (rischio cyber attack)
        - Mancanza di competenze su AI/ML nel team (gap strategico)
        """
    }
]

print("Documenti finanziari creati:")
for i, doc in enumerate(financial_documents, 1):
    print(f"{i}. {doc['title']} ({len(doc['content'])} caratteri)")
print(f"\n✅ Totale: {len(financial_documents)} documenti pronti per l'analisi")

documents = [
    Document(text=d["content"], metadata={"title": d["title"]})
    for d in financial_documents
]

# ===============================
# LLAMAINDEX SETUP
# ===============================
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)

# ===============================
# FUNZIONE DI PARSING
# ===============================
def extract_json_from_response(response_text):
    """
    Estrae JSON da una risposta che potrebbe contenere testo aggiuntivo
    """
    # Prova a trovare un blocco JSON nel testo
    json_pattern = r'\{[\s\S]*\}'
    matches = re.findall(json_pattern, response_text)

    if matches:
        for match in matches:
            try:
                return json.loads(match)
            except json.JSONDecodeError:
                continue

    # Se non trova JSON valido, restituisce dati di fallback
    return None

# ===============================
# AGENT CON FALLBACK
# ===============================
def agent_dual_output():
    """
    Sistema multi-step per ottenere output più accurati:
    1. Prima query: estrae dati storici e trend
    2. Seconda query: genera il report esecutivo
    3. Combina i risultati
    """

    # STEP 1: Estrazione dati strutturati con esempi concreti
    data_extraction_prompt = """
Sei un analista finanziario. Estrai dai documenti i seguenti dati e restituisci SOLO un JSON valido.

IMPORTANTE: Devi creare trend storici realistici basandoti sui dati Q2 2024 e stimando Q1 2024 e Q4 2023.

ESEMPIO DI OUTPUT ATTESO:
{
  "kpi": {
    "LCR": 115,
    "DebtEquity": 2.1,
    "EBITDA_Margin": 21,
    "OverdueReceivables": 19.2
  },
  "trends": {
    "quarters": ["Q4 2023", "Q1 2024", "Q2 2024"],
    "LCR": [125, 120, 115],
    "TotalDebt": [75.0, 82.0, 87.5]
  },
  "risk_matrix": [
    {"risk": "Liquidità", "probability": 4, "impact": 5},
    {"risk": "Crediti Scaduti", "probability": 4, "impact": 4},
    {"risk": "Leva Finanziaria", "probability": 3, "impact": 4},
    {"risk": "Esposizione Valutaria", "probability": 3, "impact": 3},
    {"risk": "Concentrazione Clienti", "probability": 3, "impact": 4},
    {"risk": "Tassi Interesse", "probability": 4, "impact": 3},
    {"risk": "Rischio Operativo", "probability": 3, "impact": 3}
  ]
}

REGOLE:
1. trends.quarters DEVE contenere esattamente 3 trimestri: ["Q4 2023", "Q1 2024", "Q2 2024"]
2. trends.LCR DEVE contenere 3 valori che mostrano il deterioramento: es. [125, 120, 115]
3. trends.TotalDebt DEVE contenere 3 valori che mostrano l'aumento: partendo da circa 75M a 87.5M
4. risk_matrix DEVE contenere almeno 7 rischi diversi
5. probability e impact vanno da 1 a 5

DATI DAI DOCUMENTI:
- LCR attuale: 115% (era 125% prima)
- Debito Q2: €87.5M (aumentato 18% da Q1, quindi Q1 era ~74M)
- EBITDA Margin: 21%
- Crediti scaduti >90gg: 19.2%
- Debt/Equity: 2.1

Ora genera il JSON:
"""

    # STEP 2: Generazione report testuale
    report_prompt = """
Sei il Chief Risk Officer di FinSecure Analytics. Scrivi un report esecutivo conciso (max 300 parole)
che riassuma i rischi principali identificati nei documenti.

STRUTTURA:
1. Criticità Principali (3-4 punti)
2. Raccomandazioni Prioritarie (3-4 azioni)

Usa un tono professionale ma diretto. Evidenzia numeri concreti e soglie critiche.
"""

    try:
        # Esegui entrambe le query
        console.print("[cyan]Estrazione dati strutturati...[/cyan]")
        data_response = query_engine.query(data_extraction_prompt).response
        # console.print(f"\n[yellow]Risposta dati:[/yellow]\n{data_response[:500]}...\n")

        console.print("[cyan]Generazione report esecutivo...[/cyan]")
        report_response = query_engine.query(report_prompt).response
        # console.print(f"\n[yellow]Report generato:[/yellow]\n{report_response[:300]}...\n")

        parsed_data = extract_json_from_response(data_response)

        if parsed_data and "kpi" in parsed_data and "trends" in parsed_data:
            # Validazione e correzione dei dati
            parsed_data = validate_and_fix_data(parsed_data)

            return {
                "text_report": report_response,
                "dashboard_data": parsed_data
            }
        else:
            console.print("[yellow]⚠️ Parsing JSON fallito, uso dati fallback migliorati[/yellow]")
            return generate_smart_fallback(report_response)

    except Exception as e:
        console.print(f"[red]Errore nell'agent: {e}[/red]")
        return generate_smart_fallback()


def validate_and_fix_data(data):
    """
    Valida e corregge i dati JSON per assicurare coerenza
    """
    # Assicura che trends abbia 3 trimestri
    if "trends" in data:
        if len(data["trends"].get("quarters", [])) < 3:
            data["trends"]["quarters"] = ["Q4 2023", "Q1 2024", "Q2 2024"]
            data["trends"]["LCR"] = [125, 120, 115]
            data["trends"]["TotalDebt"] = [75.0, 82.0, 87.5]

        # Assicura che le liste abbiano la stessa lunghezza
        n = len(data["trends"]["quarters"])
        if len(data["trends"].get("LCR", [])) != n:
            data["trends"]["LCR"] = [125, 120, 115][:n]
        if len(data["trends"].get("TotalDebt", [])) != n:
            data["trends"]["TotalDebt"] = [75.0, 82.0, 87.5][:n]

    # Assicura almeno 6 rischi nella matrice
    if "risk_matrix" in data:
        if len(data["risk_matrix"]) < 6:
            base_risks = [
                {"risk": "Liquidità", "probability": 4, "impact": 5},
                {"risk": "Crediti Scaduti", "probability": 4, "impact": 4},
                {"risk": "Leva Finanziaria", "probability": 3, "impact": 4},
                {"risk": "Esposizione Valutaria", "probability": 3, "impact": 3},
                {"risk": "Concentrazione Clienti", "probability": 3, "impact": 4},
                {"risk": "Tassi Interesse", "probability": 4, "impact": 3},
                {"risk": "Rischio Operativo", "probability": 3, "impact": 3}
            ]
            # Mantieni i rischi esistenti e aggiungi quelli mancanti
            existing_risks = {r["risk"]: r for r in data["risk_matrix"]}
            data["risk_matrix"] = []
            for risk in base_risks:
                data["risk_matrix"].append(existing_risks.get(risk["risk"], risk))

    return data

# ===============================
# GENERAZIONE DATI FALLBACK
# ===============================
def generate_smart_fallback(report_text=None):
    """
    Genera dati di alta qualità basati sui documenti quando l'LLM fallisce
    """
    if report_text is None:
        report_text = """
## EXECUTIVE RISK REPORT - Q2 2024

### CRITICITÀ PRINCIPALI

**1. RISCHIO LIQUIDITÀ (Critico)**
- LCR al 115% (sotto soglia regolamentare 125%)
- Trend in deterioramento: da 125% (Q4 2023) a 115% (Q2 2024)
- Necessario buffer addizionale di €15M entro 60 giorni

**2. RISCHIO CREDITO (Alto)**
- Crediti scaduti >90 giorni: €8.7M (19.2% del totale)
- Tre esposizioni critiche per €37.5M (Cliente A, B, C)
- Accantonamenti insufficienti: €2.1M vs €4.5M raccomandati

**3. LEVA FINANZIARIA (Alto)**
- Ratio Debito/Equity: 2.1 (sopra soglia critica 2.0)
- Debito cresciuto da €75M (Q4 2023) a €87.5M (Q2 2024)
- 74% del debito a tasso variabile esposto a rialzo tassi

**4. CONCENTRAZIONE RISCHI**
- Top 3 clienti: 67% dei ricavi (rischio concentrazione)
- Esposizione valutaria non coperta: 60% (€25.2M a rischio)
- Single point of failure su infrastruttura IT

### RACCOMANDAZIONI PRIORITARIE

1. **Immediato (0-30 giorni)**: Implementare buffer liquidità €15M
2. **Breve termine (30-60 giorni)**: Aumentare accantonamenti crediti a €4.5M
3. **Medio termine (60-90 giorni)**: Attivare hedging su 50% esposizione valutaria
4. **Strategico (90+ giorni)**: Piano diversificazione portafoglio clienti
        """

    return {
        "text_report": report_text,
        "dashboard_data": {
            "kpi": {
                "LCR": 115,
                "DebtEquity": 2.1,
                "EBITDA_Margin": 21,
                "OverdueReceivables": 19.2
            },
            "trends": {
                "quarters": ["Q4 2023", "Q1 2024", "Q2 2024"],
                "LCR": [125, 120, 115],
                "TotalDebt": [75.0, 82.0, 87.5]
            },
            "risk_matrix": [
                {"risk": "Liquidità", "probability": 4, "impact": 5},
                {"risk": "Crediti Scaduti", "probability": 4, "impact": 4},
                {"risk": "Leva Finanziaria", "probability": 3, "impact": 4},
                {"risk": "Esposizione Valutaria", "probability": 3, "impact": 3},
                {"risk": "Concentrazione Clienti", "probability": 3, "impact": 4},
                {"risk": "Tassi Interesse", "probability": 4, "impact": 3},
                {"risk": "Rischio Operativo", "probability": 3, "impact": 3},
                {"risk": "Compliance", "probability": 3, "impact": 4}
            ]
        }
    }

# ===============================
# OUTPUT TESTUALE
# ===============================
def print_section(title, content):
    console.print(f"\n[bold blue]--- {title} ---[/bold blue]")
    console.print(Markdown(content))

# ===============================
# MAIN
# ===============================
if __name__ == "__main__":
    console.rule("[bold red]FinSecure AI Risk Management System")

    agent_output = agent_dual_output()


    print_section(" EXECUTIVE AI RISK REPORT", agent_output["text_report"])


    data = agent_output["dashboard_data"]
    kpi = data["kpi"]
    trends = data["trends"]
    risks = data["risk_matrix"]


    # KPI CARDS
    fig_kpi = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}]*4])

    fig_kpi.add_trace(go.Indicator(
        mode="number+delta",
        value=kpi["LCR"],
        delta={"reference": 125, "valueformat": ".0f"},
        title={"text": "LCR (%)"},
        number={"suffix": "%"}
    ), 1, 1)

    fig_kpi.add_trace(go.Indicator(
        mode="number+delta",
        value=kpi["DebtEquity"],
        delta={"reference": 2.0, "valueformat": ".1f"},
        title={"text": "Debt / Equity"}
    ), 1, 2)

    fig_kpi.add_trace(go.Indicator(
        mode="number",
        value=kpi["EBITDA_Margin"],
        title={"text": "EBITDA Margin (%)"},
        number={"suffix": "%"}
    ), 1, 3)

    fig_kpi.add_trace(go.Indicator(
        mode="number",
        value=kpi["OverdueReceivables"],
        title={"text": "Receivables >90d (%)"},
        number={"suffix": "%"}
    ), 1, 4)

    fig_kpi.update_layout(
        title="📊 Executive Risk Indicators",
        height=300,
        showlegend=False
    )
    fig_kpi.show()


    # TREND
    fig_trend = make_subplots(
        rows=1, cols=2,
        subplot_titles=("LCR Trend (%)", "Total Debt (€M)")
    )

    fig_trend.add_trace(go.Scatter(
        x=trends["quarters"],
        y=trends["LCR"],
        mode="lines+markers",
        name="LCR",
        line=dict(color="red", width=3),
        marker=dict(size=10)
    ), 1, 1)

    # Linea soglia critica
    fig_trend.add_hline(
        y=125,
        line_dash="dash",
        line_color="green",
        annotation_text="Soglia Min (125%)",
        row=1, col=1
    )

    fig_trend.add_trace(go.Bar(
        x=trends["quarters"],
        y=trends["TotalDebt"],
        name="Total Debt",
        marker_color="orange"
    ), 1, 2)

    fig_trend.update_layout(
        title="📈 Risk Trend Monitoring",
        height=400,
        showlegend=True
    )
    fig_trend.show()


    # RISK MATRIX
    fig_risk = go.Figure()

    colors = ['red' if r['probability'] * r['impact'] > 12 else
              'orange' if r['probability'] * r['impact'] > 8 else 'yellow'
              for r in risks]

    fig_risk.add_trace(go.Scatter(
        x=[r["probability"] for r in risks],
        y=[r["impact"] for r in risks],
        text=[r["risk"] for r in risks],
        mode="markers+text",
        textposition="top center",
        marker=dict(
            size=[r["probability"] * r["impact"] * 2 for r in risks],
            color=colors,
            opacity=0.7,
            line=dict(width=2, color='white')
        ),
        textfont=dict(size=10, color="black")
    ))

    # Zone di rischio
    fig_risk.add_shape(type="rect", x0=0, y0=0, x1=2, y1=2,
                       fillcolor="green", opacity=0.1, line_width=0)
    fig_risk.add_shape(type="rect", x0=3, y0=3, x1=5, y1=5,
                       fillcolor="red", opacity=0.1, line_width=0)

    fig_risk.update_layout(
        title="⚠️ Risk Matrix (Probability vs Impact)",
        xaxis_title="Probability (1–5)",
        yaxis_title="Impact (1–5)",
        xaxis=dict(range=[0, 6], dtick=1),
        yaxis=dict(range=[0, 6], dtick=1),
        height=600,
        showlegend=False
    )

    fig_risk.show()

    console.print("\n[bold green]✅ Analisi completata![/bold green]")

/tmp/ipython-input-3016929606.py:34: DeprecationWarning:

Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.



╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Gemini LLM attivo                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Documenti finanziari creati:
1. Q2 2024 - Report Finanziario Trimestrale (884 caratteri)
2. Risk Assessment - Analisi Credito Q2 2024 (1088 caratteri)
3. Market Risk Report - Analisi Volatilità Mercati (1028 caratteri)
4. Compliance Report - Audit Normativo Q2 2024 (1268 caratteri)
5. Operational Risk Assessment - Q2 2024 (1043 caratteri)

✅ Totale: 5 documenti pronti per l'analisi


─────────────────────────────────────── FinSecure AI Risk Management System ───────────────────────────────────────

Estrazione dati strutturati...

Generazione report esecutivo...

---  EXECUTIVE AI RISK REPORT ---

Report Esecutivo del Chief Risk Officer - Q2 2024                                                                  

Il presente report riassume le principali criticità di rischio emerse nel secondo trimestre 2024, evidenziando aree
che richiedono un'azione immediata per salvaguardare la stabilità e la conformità aziendale.                       

Criticità Principali:                                                                                              

 • Rischio di Liquidità e Non-Conformità Normativa: L'azienda opera con un Liquidity Coverage Ratio (LCR) del 115% 
   (minimo richiesto 125%) e un indice di liquidità corrente di 1.15 (soglia regolamentare 1.25), risultando non   
   conforme a Basilea III. Questa situazione è aggravata da ritardi nei pagamenti da parte di clienti chiave per   
   €8.2M e dalla mancanza di una policy formale di gestione del rischio di liquidità.                              
 • Elevato Rischio Operativo e Tecnologico: Si è registrato un aumento degli incidenti operativi (12 vs 7 in Q1)   
   con perdite totali di €385K, e un downtime dei sistemi IT di 14 ore (SLA massimo 8 ore). Permangono             
   vulnerabilità significative, tra cui un single point of failure sul sistema core banking, l'assenza di backup   
   geografico dei dati e solo il 30% della formazione sulla cyber security completata. Il Business Continuity Plan 
   non è aggiornato dal 2022.                                                                                      
 • Debolezze Finanziarie e di Governance: Il rapporto debito/equity ha raggiunto 2.1 (soglia critica 2.0), con un  
   aumento del 18% dell'esposizione debitoria. La concentrazione dei ricavi è elevata, con i primi 3 clienti che   
   rappresentano il 67%. Il turnover del personale chiave è al 18% (soglia critica 12%). Mancano procedure formali 
   per il change management IT, l'escalation dei rischi operativi e un sistema di Three Lines of Defense.          

Raccomandazioni Prioritarie:                                                                                       

 • Rafforzare la Posizione di Liquidità e la Conformità: Implementare un buffer di liquidità addizionale di €15M   
   entro 60 giorni e completare l'upgrade del sistema di impairment crediti entro il Q3 2024. Nominare urgentemente
   un Compliance Officer dedicato.                                                                                 
 • Migliorare la Resilienza Operativa e IT: Aggiornare immediatamente il Business Continuity Plan e formalizzare le
   procedure di change management IT e di escalation dei rischi operativi. Completare la formazione sulla cyber    
   security e schedulare test di disaster recovery.                                                                
 • Potenziare la Governance del Rischio: Formalizzare l'implementazione del sistema delle Three Lines of Defense e 
   schedulare stress test mensili a partire dal Q3 2024. Sviluppare strategie per mitigare la concentrazione       
   clienti e la dipendenza da fornitori unici.

✅ Analisi completata!